# Model Training Pipeline

## 1. Project Setup and Reproducibility Controls

This pipeline compares fragment-intensity models using training and validation data only. Singly charged b/y ions and precursor charges 2, 3, and 4 define the prediction task. Acquisition labels do not, by themselves, establish CID provenance; this must be verified independently before interpreting results as CID-specific.

Select the project `.venv` kernel. Named factories implement `forward(tokens, charge, frag=None)` and return the common 58-slot intensity vector. A small adapter may provide this interface for other models.

In [16]:
import sys
from pathlib import Path
import hashlib
import json
import math
import tempfile
import uuid
from dataclasses import dataclass, asdict, replace
from datetime import datetime, timezone
from itertools import product

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd().resolve()
if ROOT.name == "refac":
    ROOT = ROOT.parent
if not (ROOT / "train_deep.py").exists():
    raise RuntimeError("Run from the project root or refac directory.")
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Select the project .venv Python kernel.")
sys.path.insert(0, str(ROOT))
import common as c
from train_deep import PrositLike, encode, masked_spectral_angle_loss, set_seed, pick_device, git_commit
from refac.models import PeptideEncoder

DEVICE = pick_device()
GIT_COMMIT = git_commit()
print("Python:", sys.executable)
print("Device:", DEVICE)

Python: /Users/fridolin.karger/dev/sommer/.venv/bin/python
Device: mps


## 2. Typed Configuration for Run Modes, Paths, and Search Space

`mode` controls the experimental budget rather than model architecture. Preliminary runs sample fixed subsets and use a short epoch budget. Full runs use every eligible training row and the complete eligible validation split. Charges outside 2, 3, and 4 are excluded explicitly. The configuration below does not initiate training. Search strategies supply dictionaries of constructor arguments and training overrides.

In [17]:
@dataclass(frozen=True)
class PipelineConfig:
    mode: str = "preliminary"
    data_dir: Path = ROOT / "refac"
    output_dir: Path = ROOT / "refac"
    batch_size: int = 256
    max_epochs: int = 50
    patience: int = 5
    lr: float = 1e-3
    weight_decay: float = 1e-5
    preliminary_train: int = 2000
    preliminary_val: int = 500
    preliminary_epochs: int = 2
    seed: int = 1312
    cache: bool = True

    def __post_init__(self):
        if self.mode not in {"preliminary", "full"}:
            raise ValueError("mode must be preliminary or full")
        for name in ("batch_size", "max_epochs", "patience", "preliminary_train", "preliminary_val", "preliminary_epochs"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name} must be positive")
        if self.lr <= 0 or self.weight_decay < 0:
            raise ValueError("lr must be positive and weight_decay nonnegative")


@dataclass(frozen=True)
class SearchConfig:
    strategy: str = "grid"
    trials: int = 4
    seed: int = 1312


CHARGES = (2, 3, 4)
ARRAY_KEYS = ("tokens", "charge", "frag", "Y", "mask")
SCORE_COLUMNS = ["model", "params", "val SA", "z=2", "z=3", "z=4", "checkpoint"]
CONFIG = PipelineConfig()


def config_record(config):
    return {name: str(value) if isinstance(value, Path) else value
            for name, value in asdict(config).items()}

## 3. Model Registry Abstraction

Each registry entry contains a builder, default constructor arguments, and a forward-call adapter. New PyTorch models require only one entry; the training loop does not inspect architecture names. For a two-input model, supply an adapter such as `lambda model, tokens, charge, frag: model(tokens, charge)`. Checkpoints store constructor arguments and a state dictionary rather than pickled model objects; restore the same registry entry before loading.

In [18]:
def forward_with_frag(model, tokens, charge, frag):
    return model(tokens, charge, frag)


MODELS = {
    "PrositLike": {
        "build": PrositLike,
        "defaults": {"emb": 64, "hidden": 128, "head": "pooled"},
        "forward": forward_with_frag,
    },
    "PeptideEncoder": {
        "build": PeptideEncoder,
        "defaults": {"aa_vocab_size": 21, "mod_vocab_size": 2, "precursor_vocab_size": 3,
                     "hidden_dimensions": 256, "transformer_layers": 6, "attention_heads": 8},
        "forward": forward_with_frag,
    },
}


def build_registered_model(name, overrides=None, device=DEVICE):
    entry = MODELS[name]
    parameters = entry["defaults"] | (overrides or {})
    return entry["build"](**parameters).to(device), parameters


print("Registered models:", ", ".join(MODELS))

Registered models: PrositLike, PeptideEncoder


## 4. Train/Validation Data Loader

Only the dedicated `clean_spectra_train_split.parquet` and `clean_spectra_val_split.parquet` files are permitted. Parquet predicates restrict rows to the expected split before materialization; test paths and split names are never requested. Charges outside the predefined policy are excluded, and unsupported peptide lengths cause an error rather than silent truncation. Encoded caches are keyed by source-file metadata, encoding-helper content, and charge policy. Source Parquet files remain unchanged.

In [19]:
def load_split(config, split):
    if split not in {"train", "val"}:
        raise ValueError("Only train and val splits are permitted")
    filename = f"clean_spectra_{split}_split.parquet"
    path = (Path(config.data_dir) / filename).resolve()
    if path.name != filename:
        raise ValueError("Source must resolve to the expected train/validation filename")
    source = path.stat()
    signature = hashlib.sha256()
    signature.update(f"v1:{path}:{source.st_size}:{source.st_mtime_ns}:{CHARGES}".encode())
    for helper in (ROOT / "common.py", ROOT / "train_deep.py"):
        signature.update(helper.read_bytes())
    cache_path = Path(config.output_dir) / "cache" / f"pipeline_{split}_{signature.hexdigest()[:16]}.npz"
    if config.cache and cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as cached:
            return {key: cached[key] for key in ARRAY_KEYS}
    columns = ["raw_file", "scan_number", "peptide_sequence", "precursor_charge",
               "frag", "matched_ions", "intensities_raw", "split"]
    frame = pd.read_parquet(path, columns=columns, filters=[("split", "==", split)])
    if frame.empty or not frame["split"].eq(split).all():
        raise ValueError(f"Expected a nonempty {split} split")
    excluded = int((~frame["precursor_charge"].isin(CHARGES)).sum())
    frame = frame.loc[frame["precursor_charge"].isin(CHARGES)].copy()
    if frame.empty:
        raise ValueError(f"No eligible rows in {split}")
    lengths = frame["peptide_sequence"].map(c.peptide_length)
    if not lengths.between(2, c.MAX_PEPTIDE_LEN).all():
        raise ValueError("Peptide lengths must be between 2 and 30; truncation is not permitted")
    frame = frame.sort_values(["raw_file", "scan_number"]).reset_index(drop=True)
    arrays = encode(frame)
    if config.cache:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, **arrays)
    print(f"{split}: {len(frame):,} eligible rows; {excluded:,} excluded by charge policy")
    return arrays


def load_data(config):
    return {split: load_split(config, split) for split in ("train", "val")}


def data_fingerprint(data):
    digest = hashlib.sha256()
    for split in ("train", "val"):
        for key in ARRAY_KEYS:
            values = np.ascontiguousarray(data[split][key])
            digest.update(f"{split}:{key}:{values.shape}:{values.dtype}".encode())
            digest.update(values.tobytes())
    return digest.hexdigest()[:16]


try:
    load_split(CONFIG, "test")
except ValueError:
    print("Split guard passed: disallowed split rejected before file access")
else:
    raise AssertionError("Disallowed split was accepted")

Split guard passed: disallowed split rejected before file access


## 5. Unified Trainer Interface

One loop handles all registered models. Mini-batches move from CPU memory to the selected device, avoiding whole-dataset accelerator allocation. Adam optimization and validation-driven learning-rate reduction are shared across architectures. Early stopping selects the highest validation spectral angle, and the returned weights are restored to that epoch. Reported SA follows the existing convention in `common.evaluate_vectors`, including clipping predictions to [0, 1]. Model outputs must be finite matrices matching the target layout.

In [20]:
def select_rows(arrays, limit, seed):
    size = len(arrays["tokens"])
    if not size or not np.isin(arrays["charge"], CHARGES).all():
        raise ValueError("Expected nonempty arrays with charges 2, 3, 4 only")
    indices = np.arange(size)
    if limit is not None and size > limit:
        indices = np.sort(np.random.default_rng(seed).choice(size, limit, replace=False))
    return {key: arrays[key][indices] for key in ARRAY_KEYS}


def make_loader(arrays, config, shuffle=False):
    tensors = [torch.from_numpy(np.array(arrays[key], copy=True, order="C")) for key in ARRAY_KEYS]
    return DataLoader(TensorDataset(*tensors), batch_size=config.batch_size,
                      shuffle=shuffle, generator=torch.Generator(device="cpu").manual_seed(config.seed))


def forward_batch(name, model, batch, device):
    tokens, charge, frag, target, mask = [values.to(device) for values in batch]
    prediction = MODELS[name]["forward"](model, tokens, charge, frag)
    if prediction.shape != target.shape or not torch.isfinite(prediction).all().item():
        raise ValueError(f"{name} must return finite predictions with shape {tuple(target.shape)}")
    return prediction, target, mask


@torch.inference_mode()
def predict_arrays(name, model, loader, device):
    model.eval()
    return np.concatenate([forward_batch(name, model, batch, device)[0].cpu().numpy()
                           for batch in loader])


def train_one(name, data, config, model_params=None, device=DEVICE):
    set_seed(config.seed)
    started = datetime.now(timezone.utc)
    run_id = f"{started:%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:8]}"
    preliminary = config.mode == "preliminary"
    train = select_rows(data["train"], config.preliminary_train if preliminary else None, config.seed)
    val = select_rows(data["val"], config.preliminary_val if preliminary else None, config.seed)
    train_loader = make_loader(train, config, shuffle=True)
    val_loader = make_loader(val, config)
    model, parameters = build_registered_model(name, model_params, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=1)
    epochs = min(config.max_epochs, config.preliminary_epochs) if preliminary else config.max_epochs
    best_score, best_state, best_epoch, stale = -math.inf, None, 0, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            prediction, target, mask = forward_batch(name, model, batch, device)
            loss = masked_spectral_angle_loss(prediction, target, mask)
            if not torch.isfinite(loss).item():
                raise ValueError("Non-finite training loss")
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(target)
        prediction = predict_arrays(name, model, val_loader, device)
        metrics = validation_metrics(prediction, val)
        score = metrics["val SA"]
        if not math.isfinite(score):
            raise ValueError("Non-finite validation score")
        scheduler.step(1 - score)
        history.append({"epoch": epoch, "train_loss": loss_sum / len(train["tokens"]), "val SA": score})
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            stale += 1
        print(f"{name} [{config.mode}] epoch {epoch}: loss={history[-1]['train_loss']:.4f}, val SA={score:.4f}")
        if stale >= config.patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    final_metrics = validation_metrics(predict_arrays(name, model, val_loader, device), val)
    record = {
        "run_id": run_id, "timestamp": started.isoformat(), "model": name, "mode": config.mode,
        "params": sum(parameter.numel() for parameter in model.parameters()), **final_metrics,
        "model_params": parameters, "config": config_record(config), "git_commit": GIT_COMMIT,
        "dataset": data_fingerprint(data), "n_train": len(train["tokens"]), "n_val": len(val["tokens"]),
        "epochs trained": len(history), "best_epoch": best_epoch, "checkpoint": None,
        "seconds": (datetime.now(timezone.utc) - started).total_seconds(), "history": history,
    }
    return model, record

## 6. Preliminary Run Mode

Preliminary experiments use reproducibly sampled training and validation subsets. Each model is freshly initialized with the same seed, trained for at most `preliminary_epochs`, and reported in a temporary in-memory table. These scores are exploratory and never enter the final leaderboard. Define functions now; use the execution controls at the end after all definitions have been evaluated.

In [21]:
def run_preliminary(experiments, data, config=CONFIG, device=DEVICE):
    return run_pipeline(experiments, data, replace(config, mode="preliminary"), device)

## 7. Full Run Mode

Full experiments retain all training rows allowed by the charge policy and evaluate the entire eligible validation split. Validation remains excluded from gradient updates. The saved model is the best validation epoch, not necessarily the last epoch before early stopping. Every completed full run is persisted immediately, so subsequent model failures do not discard earlier results. `experiments` maps registered model names to constructor overrides.

In [22]:
def run_pipeline(experiments, data, config=CONFIG, device=DEVICE):
    records = []
    for name, parameters in experiments.items():
        model, record = train_one(name, data, config, parameters, device)
        if config.mode == "full":
            save_final(model, record, config)
        records.append(record)
        del model
    if not records:
        raise ValueError("Provide at least one model experiment")
    if config.mode == "full":
        leaderboard(config.output_dir, dataset=records[0]["dataset"])
    else:
        table = pd.DataFrame(records).sort_values("val SA", ascending=False)
        print("Preliminary results (not final benchmarks)")
        print(table[SCORE_COLUMNS].to_string(index=False))
    return records

## 8. Evaluation Block

Validation spectral angle is averaged per spectrum, both overall and separately for precursor charges 2, 3, and 4. An absent charge group is represented by `None`, not by an artificial zero. Scores describe validation performance under model selection and are not independent generalization estimates. Parameter counts are recorded separately by the trainer.

In [23]:
def validation_metrics(prediction, arrays):
    if prediction.shape != arrays["Y"].shape or not np.isfinite(prediction).all():
        raise ValueError("Validation predictions must be finite and match target shape")
    target, mask = arrays["Y"], arrays["mask"].astype(bool)
    metrics = {"val SA": c.evaluate_vectors(target, prediction, mask)["spectral_angle"]}
    for charge in CHARGES:
        selected = arrays["charge"] == charge
        metrics[f"z={charge}"] = (
            c.evaluate_vectors(target[selected], prediction[selected], mask[selected])["spectral_angle"]
            if selected.any() else None
        )
    return metrics

## 9. Result Persistence

Full runs save a reconstructable best-epoch checkpoint under `refac/checkpoints` and append a summary to `refac/results/pipeline_runs.jsonl`. The checkpoint includes the state dictionary, constructor arguments, configuration, training history, and dataset fingerprint. The summary retains the required comparison fields and provenance without epoch-level history. Preliminary runs cannot be saved through this function. Restoring a custom architecture requires registering its builder and adapter first.

In [24]:
def append_record(path, record):
    payload = json.dumps(record, allow_nan=False)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as stream:
        stream.write(payload + "\n")


def save_final(model, record, config):
    if config.mode != "full" or record["mode"] != "full":
        raise ValueError("Only full runs may produce final benchmark artifacts")
    checkpoint = Path(config.output_dir).resolve() / "checkpoints" / f"pipeline_{record['run_id']}.pt"
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    record["checkpoint"] = str(checkpoint)
    summary = {key: value for key, value in record.items() if key != "history"}
    json.dumps(summary, allow_nan=False)
    temporary = checkpoint.with_suffix(".tmp")
    torch.save({"state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
                "record": record}, temporary)
    temporary.replace(checkpoint)
    append_record(Path(config.output_dir) / "results" / "pipeline_runs.jsonl", summary)
    return checkpoint


def restore_model(checkpoint, device=DEVICE):
    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
    record = saved["record"]
    model, _ = build_registered_model(record["model"], record["model_params"], device)
    model.load_state_dict(saved["state_dict"])
    return model.eval(), record

## 10. Hyperparameter Search Abstraction

Search strategies generate parameter dictionaries and do not own training logic. Grid search enumerates the finite Cartesian product; random search samples up to `trials` distinct combinations reproducibly without constructing the complete grid. Register additional generators in `SEARCH_STRATEGIES` with the same `(space, search_config)` interface. Trials always use preliminary mode and identical sampled data. Their summaries are stored separately in `pipeline_trials.jsonl`; selected settings must receive a full run before entering the final leaderboard. Repeated validation-based selection can produce optimistic estimates.

In [25]:
def grid_strategy(space, search):
    for values in product(*space.values()):
        yield dict(zip(space, values))


def random_strategy(space, search):
    if search.trials < 1:
        raise ValueError("trials must be positive")
    total = math.prod(len(values) for values in space.values())
    indices = np.random.default_rng(search.seed).choice(total, min(total, search.trials), replace=False)
    for index in indices:
        candidate = {}
        for key, options in reversed(list(space.items())):
            index, offset = divmod(int(index), len(options))
            candidate[key] = options[offset]
        yield candidate


SEARCH_STRATEGIES = {"grid": grid_strategy, "random": random_strategy}


def search_model(name, data, model_space, training_space=None, config=CONFIG,
                 search=SearchConfig(), device=DEVICE):
    training_space = training_space or {}
    allowed = {"lr", "weight_decay", "batch_size", "max_epochs", "patience"}
    if not set(training_space).issubset(allowed):
        raise ValueError(f"Training search supports only {sorted(allowed)}")
    space = {(group, key): list(values)
             for group, entries in (("model", model_space), ("training", training_space))
             for key, values in entries.items()}
    if any(not values for values in space.values()):
        raise ValueError("Search dimensions must contain at least one value")
    search_id = uuid.uuid4().hex
    records = []
    for candidate in SEARCH_STRATEGIES[search.strategy](space, search):
        model_parameters = {key: value for (group, key), value in candidate.items() if group == "model"}
        training_overrides = {key: value for (group, key), value in candidate.items() if group == "training"}
        trial_config = replace(config, mode="preliminary", **training_overrides)
        model, record = train_one(name, data, trial_config, model_parameters, device)
        record = {key: value for key, value in record.items() if key != "history"}
        record.update(search_id=search_id, training_overrides=training_overrides)
        append_record(Path(config.output_dir) / "results" / "pipeline_trials.jsonl", record)
        records.append(record)
        del model
    if not records:
        raise ValueError("Search strategy produced no candidates")
    return pd.DataFrame(records).sort_values("val SA", ascending=False).reset_index(drop=True)


example_space = {("model", "hidden"): [16, 32], ("training", "lr"): [1e-3, 1e-4]}
assert len(list(grid_strategy(example_space, SearchConfig()))) == 4
random_config = SearchConfig(strategy="random", trials=3)
assert list(random_strategy(example_space, random_config)) == list(random_strategy(example_space, random_config))
print("Search generators passed: grid coverage and reproducible random sampling")

Search generators passed: grid coverage and reproducible random sampling


## 11. Model Leaderboard Generation

The leaderboard reads only completed full-run summaries and orders runs by descending validation SA. Multiple runs of a model remain visible rather than silently retaining only a favorable seed. Dataset fingerprints must match; if multiple datasets are present, select one explicitly. Preliminary and search results are deliberately excluded. The requested compact columns are printed, while the returned table retains configuration and provenance.

In [26]:
def leaderboard(output_dir=ROOT / "refac", dataset=None):
    path = Path(output_dir) / "results" / "pipeline_runs.jsonl"
    if not path.exists():
        print("No completed full runs yet.")
        return pd.DataFrame(columns=SCORE_COLUMNS)
    with path.open() as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    records = [record for record in records if record["mode"] == "full"]
    if dataset is None and len({record["dataset"] for record in records}) > 1:
        raise ValueError("Multiple datasets found; provide dataset=<fingerprint> for a comparable leaderboard")
    if dataset is not None:
        records = [record for record in records if record["dataset"] == dataset]
    if not records:
        print("No matching full runs.")
        return pd.DataFrame(columns=SCORE_COLUMNS)
    table = pd.DataFrame(records).sort_values("val SA", ascending=False, kind="stable").reset_index(drop=True)
    print(table[SCORE_COLUMNS].to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    return table


final_results = leaderboard(CONFIG.output_dir)

No completed full runs yet.


In [27]:
long_config = replace(
    CONFIG,
    mode="full",
    max_epochs=100,
    patience=15,
    batch_size=128,
    lr=0.0006038585076587369,
    weight_decay=1.8421828349040193e-06,
    seed=1312,
)

experiments = {
    "PeptideEncoder": {
        "hidden_dimensions": 256,
        "transformer_layers": 6,
        "attention_heads": 8,
    },
}

data = load_data(long_config)
records = run_pipeline(experiments, data, long_config)

print("Best epoch:", records[0]["best_epoch"])
print("Validation SA:", records[0]["val SA"])
print("Checkpoint:", records[0]["checkpoint"])

train: 279,832 eligible rows; 0 excluded by charge policy
val: 34,979 eligible rows; 0 excluded by charge policy


/Users/fridolin.karger/dev/sommer/baukasten.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


PeptideEncoder [full] epoch 1: loss=0.2324, val SA=0.7606
PeptideEncoder [full] epoch 2: loss=0.1910, val SA=0.7705
PeptideEncoder [full] epoch 3: loss=0.1812, val SA=0.7854
PeptideEncoder [full] epoch 4: loss=0.1755, val SA=0.7775
PeptideEncoder [full] epoch 5: loss=0.1715, val SA=0.7726
PeptideEncoder [full] epoch 6: loss=0.1606, val SA=0.7922
PeptideEncoder [full] epoch 7: loss=0.1583, val SA=0.7888
PeptideEncoder [full] epoch 8: loss=0.1567, val SA=0.7977
PeptideEncoder [full] epoch 9: loss=0.1554, val SA=0.7985
PeptideEncoder [full] epoch 10: loss=0.1542, val SA=0.7984
PeptideEncoder [full] epoch 11: loss=0.1532, val SA=0.7973
PeptideEncoder [full] epoch 12: loss=0.1477, val SA=0.8051
PeptideEncoder [full] epoch 13: loss=0.1463, val SA=0.8070
PeptideEncoder [full] epoch 14: loss=0.1455, val SA=0.8108
PeptideEncoder [full] epoch 15: loss=0.1449, val SA=0.8120
PeptideEncoder [full] epoch 16: loss=0.1443, val SA=0.8112
PeptideEncoder [full] epoch 17: loss=0.1436, val SA=0.8119
Peptid

## Execution Controls

Evaluate all definition cells above before using these controls. Set `CONFIG.mode` to `preliminary` or `full`, edit constructor overrides in `EXPERIMENTS`, and enable the desired operation. Both switches default to false to prevent accidental research-data access or long training runs. To promote a search result, use its `model_params` and `training_overrides` in a separate full run, for example:

```python
best = search_results.iloc[0]
full_config = replace(CONFIG, mode="full", **best["training_overrides"])
full_records = run_pipeline({"PrositLike": best["model_params"]}, data, full_config)
restored_model, metadata = restore_model(full_records[0]["checkpoint"])
```

The data loader does not change the predefined split or modify source files.

In [12]:
RUN_EXPERIMENTS = False
RUN_SEARCH = False
EXPERIMENTS = {"PrositLike": {}, "PeptideEncoder": {}}

if RUN_EXPERIMENTS or RUN_SEARCH:
    data = load_data(CONFIG)
if RUN_SEARCH:
    search_results = search_model(
        "PrositLike", data,
        model_space={"hidden": [64, 128]},
        training_space={"lr": [1e-3, 3e-4]},
        config=CONFIG,
        search=SearchConfig(strategy="random", trials=4),
    )
    print(search_results[["model", "model_params", "training_overrides", "val SA"]].to_string(index=False))
if RUN_EXPERIMENTS:
    run_records = run_pipeline(EXPERIMENTS, data, CONFIG)
if not (RUN_EXPERIMENTS or RUN_SEARCH):
    print("Research-data training is disabled. Enable an execution switch to run experiments.")

Research-data training is disabled. Enable an execution switch to run experiments.


## Synthetic Pipeline Verification

The following checks use only fabricated spectra and temporary train/validation Parquet files. They exercise charge filtering, encoded-cache reuse and invalidation, preliminary subset sizes, full-run row counts, both supplied architectures, a custom two-input model adapter, checkpoint reconstruction, charge-stratified scoring, search isolation, and leaderboard ordering. No research dataset is loaded, and temporary artifacts are removed automatically. These checks establish pipeline behavior, not predictive performance.

In [15]:
def pipeline_self_check():
    class TwoInputModel(torch.nn.Module):
        def __init__(self, width=4):
            super().__init__()
            self.embedding = torch.nn.Embedding(c.VOCAB_SIZE, width)
            self.head = torch.nn.Linear(width, c.VECTOR_DIM)

        def forward(self, tokens, charge):
            return self.head(self.embedding(tokens).mean(dim=1)).sigmoid()

    MODELS["TwoInputCheck"] = {
        "build": TwoInputModel, "defaults": {"width": 4},
        "forward": lambda model, tokens, charge, frag: model(tokens, charge),
    }
    try:
        with tempfile.TemporaryDirectory(prefix="sommer_pipeline_") as folder:
            temporary = Path(folder)
            frames = {}
            for split in ("train", "val"):
                frames[split] = pd.DataFrame({
                    "raw_file": [f"synthetic_{split}"] * 7,
                    "scan_number": list(range(7)),
                    "peptide_sequence": ["GASPK", "AC[UNIMOD:4]DM[UNIMOD:35]K", "LTQETNRV", "GASPK", "ACDK", "PEPTIDEK", "GASPK"],
                    "precursor_charge": [2, 3, 4, 2, 3, 4, 5],
                    "frag": ["DDA-1h"] * 7,
                    "matched_ions": [["b1", "y1", "b2"] for _ in range(7)],
                    "intensities_raw": [[2.0, 1.0, 0.5] for _ in range(7)],
                    "split": [split] * 7,
                })
                frames[split].to_parquet(temporary / f"clean_spectra_{split}_split.parquet", index=False)
            config = PipelineConfig(
                mode="full", data_dir=temporary, output_dir=temporary,
                batch_size=3, max_epochs=1, patience=1,
                preliminary_train=3, preliminary_val=3, preliminary_epochs=1,
            )
            synthetic = load_data(config)
            assert all(len(synthetic[split]["tokens"]) == 6 for split in ("train", "val"))
            fingerprint = data_fingerprint(synthetic)
            assert data_fingerprint(load_data(config)) == fingerprint
            preliminary = run_preliminary({"TwoInputCheck": {}}, synthetic, config, device="cpu")
            assert preliminary[0]["n_train"] == preliminary[0]["n_val"] == 3
            assert not (temporary / "results" / "pipeline_runs.jsonl").exists()
            experiments = {
                "PrositLike": {"emb": 8, "hidden": 8},
                "PeptideEncoder": {"hidden_dimensions": 16, "transformer_layers": 1, "attention_heads": 2},
            }
            records = run_pipeline(experiments, synthetic, config, device="cpu")
            for record in records:
                assert record["n_train"] == record["n_val"] == 6
                assert record["params"] > 0 and record["best_epoch"] == 1
                restored, metadata = restore_model(record["checkpoint"], device="cpu")
                prediction = predict_arrays(record["model"], restored, make_loader(synthetic["val"], config), "cpu")
                measured = validation_metrics(prediction, synthetic["val"])
                for field in ("val SA", "z=2", "z=3", "z=4"):
                    assert math.isclose(measured[field], record[field], abs_tol=1e-7)
                assert metadata["run_id"] == record["run_id"]
            trials = search_model(
                "TwoInputCheck", synthetic, {"width": [4, 8]}, {"lr": [1e-3]}, config,
                SearchConfig(strategy="random", trials=2), device="cpu",
            )
            assert len(trials) == 2 and trials["checkpoint"].isna().all()
            assert trials["mode"].eq("preliminary").all()
            assert len((temporary / "results" / "pipeline_trials.jsonl").read_text().splitlines()) == 2
            ranked = leaderboard(temporary)
            assert len(ranked) == 2 and ranked["val SA"].is_monotonic_decreasing
            assert set(SCORE_COLUMNS).issubset(ranked.columns)
            assert len(list((temporary / "checkpoints").glob("*.pt"))) == 2
            one_charge = {key: values[:1] for key, values in synthetic["val"].items()}
            missing = validation_metrics(one_charge["Y"], one_charge)
            assert missing["z=3"] is None and missing["z=4"] is None
            frames["train"].at[0, "intensities_raw"] = [1.0, 2.0, 0.5]
            frames["train"].to_parquet(temporary / "clean_spectra_train_split.parquet", index=False)
            assert data_fingerprint(load_data(config)) != fingerprint
    finally:
        MODELS.pop("TwoInputCheck", None)
    print("All synthetic pipeline checks passed; no research data were accessed.")


pipeline_self_check()

train: 6 eligible rows; 1 excluded by charge policy
val: 6 eligible rows; 1 excluded by charge policy
TwoInputCheck [preliminary] epoch 1: loss=0.5630, val SA=0.4380
Preliminary results (not final benchmarks)
        model  params   val SA      z=2      z=3      z=4 checkpoint
TwoInputCheck     382 0.437984 0.433206 0.574298 0.306448       None
PrositLike [full] epoch 1: loss=0.6271, val SA=0.3755
PeptideEncoder [full] epoch 1: loss=0.6655, val SA=0.3402
         model  params  val SA    z=2    z=3    z=4                                                                                                                         checkpoint
    PrositLike    3851  0.3755 0.4100 0.4398 0.2769 /private/var/folders/fc/y1fk3zyx12q75f3nh6n9rn300000gp/T/sommer_pipeline_zp2o_xen/checkpoints/pipeline_20261014T104328_fa70c1d6.pt
PeptideEncoder    5842  0.3402 0.3600 0.3963 0.2643 /private/var/folders/fc/y1fk3zyx12q75f3nh6n9rn300000gp/T/sommer_pipeline_zp2o_xen/checkpoints/pipeline_20261014T104328_6fde

/Users/fridolin.karger/dev/sommer/baukasten.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
/Users/fridolin.karger/dev/sommer/baukasten.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
